# Data Preparation

## Train-Test-Split

In [ ]:
# perform train-test-split
from sklearn.model_selection import train_test_split

target = df.loc[:,'IsBadBuy']
features = df.drop('IsBadBuy', axis=1)

features_train, features_test, target_train, target_test = train_test_split(features, 
                                                                            target, 
                                                                            random_state=42,
                                                                            test_size=0.1)

In [ ]:
# save features_test as 'features_test.csv'
features_test.to_csv('features_test.csv', index=False)

## Data Preparation

### Datatype Transformation

In [ ]:
# clean_data function
def clean_data(df):
    """Returns cleaned DataFrame.
    Clean dataset for both TRAIN and TEST:
       - convert PurchDate to datetime
       - drop irrelevant columns
       - replace invalid MMR values (0.0) with NaN
       - drop rows with critical NaN
    
    Args: 
        df (pd.DataFrame) : uncleaned DataFrame
        
    Returns:
        df  (pd.DataFrame) : cleaned DataFrame
    
    """
    #to datetime
    if 'PurchDate' in df.columns:
        if df['PurchDate'].dtype == 'int64':
            df['PurchDate'] = pd.to_datetime(df['PurchDate'], unit='s')
        else:
            df['PurchDate'] = pd.to_datetime(df['PurchDate'], errors='coerce')

    
    # drop columns that are not useful
    cols_to_drop = [
        'PRIMEUNIT', 'AUCGUART',               # 95% missing
        'BYRNO',                               # meaningless ID
        'Model', 'SubModel',                   # high cardinality
        'WheelTypeID',                         # keep WheelType
        'VNST',                                # keep ZIP
        'VehYear',                             # duplicate of VehicleAge
        'Trim',                                # 134 unique value, 3% missing
        'MMRAcquisitionAuctionAveragePrice',   # keep MMRCurrentRetailCleanPrice
        'MMRAcquisitionAuctionCleanPrice',
        'MMRAcquisitionRetailAveragePrice',
        'MMRAcquisitonRetailCleanPrice',
        'MMRCurrentAuctionAveragePrice',
        'MMRCurrentAuctionCleanPrice',
        'MMRCurrentRetailAveragePrice'
    ]
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')
 
    # replace invalid MMR values (0.0) with NaN
    df['MMRCurrentRetailCleanPrice'] = df['MMRCurrentRetailCleanPrice'].replace(0, np.nan)
                             
    return df

In [ ]:
features_train = clean_data(features_train)
features_test = clean_data(features_test)

In [ ]:
cat_cols = features_train.select_dtypes(include='object').columns
for col in cat_cols:
    print(col, ':', features_train.loc[:, col].unique())

In [ ]:
# normalize categorical values: unify placeholder categories in Color,
# standardizes Transmission labels and reconstructs TopThreeAmericanName as a binary feature based on the Make column

def clean_categorical_features(df):
    """"
    Cleans and normalizes selected categorical features:
    - Color: groups placeholder values into 'UNKNOWN'
    - Transmission: standardizes inconsistent casing ('Manual' into 'MANUAL')
    - TopThreeAmericanName: Reconstructs as a binary feature using Make (1 = top‑3 US brands, 0 = other)

    """
    df['Color'] = df['Color'].replace(['NOT AVAIL', 'OTHER'], 'UNKNOWN')
    df['Transmission'] = df['Transmission'].replace('Manual', 'MANUAL')
    df = df.drop(columns='TopThreeAmericanName', errors='ignore')

    top3_brands = [
    'CHEVROLET', 'GMC', 'BUICK', 'CADILLAC', 'PONTIAC', 'SATURN', 'OLDSMOBILE',  # GM
    'FORD',                                                                      # Ford
    'CHRYSLER', 'DODGE', 'JEEP'                                                  # Chrysler
    ]

    df['TopThreeAmericanName'] = df['Make'].apply(lambda x: 1 if x in top3_brands else 0)
    
    
    return df


In [ ]:
features_train = clean_categorical_features(features_train)
features_test = clean_categorical_features(features_test)

### Data Imputation

In [ ]:
def impute_missing_values(df):
    """
    Impute missing values for TRAIN/TEST datasets:
    - categorical columns to 'UNKNOWN'
    - numerical columns to median
    Does NOT remove rows and does NOT perform feature engineering.
    
    """

    df = df.copy()
   
    # fill categorical NaN
    cat_cols = df.select_dtypes(include='object').columns
    df[cat_cols] = df[cat_cols].fillna('UNKNOWN')
    
    # fill numerical NaN
    num_cols = df.select_dtypes(include='number').columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
                
    return df

In [ ]:
features_train = impute_missing_values(features_train)
features_test = impute_missing_values(features_test)

### Deal with outliers

In [ ]:
# Boxplot- Variante streng vs weak 
ct = pd.crosstab(df['IsBadBuy'], columns='count')
ct['percent'] = (ct['count'] / ct['count'].sum() * 100)
print(ct, '\n\n')


streng = 1.5
weak = 3
key_cols = ['VehOdo', 'VehBCost', 'WarrantyCost', 'MMRCurrentRetailCleanPrice', 'VehicleAge']

for col in key_cols:

    # percentiles
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    for var in [streng, weak]:

        upper = q3 + var * iqr
        lower = q1 - var * iqr

        mask = (df[col] > upper) | (df[col] < lower)

        print(f'\nOutliers for {col} with {var}*IQR:', mask.sum())
        display(pd.crosstab(df.loc[mask, 'IsBadBuy'], columns='count'))

        # plot
        plt.figure(figsize=(6,4))
        sns.scatterplot(x=df[col], y=df.index, hue=mask)
        plt.title(f'{var}*IQR Outliers for Variable {col}')
        plt.legend(title='Outlier')
        plt.show()

In [ ]:
# Notes:
# IQR outliers show a strong correlation with BadBuy, so removing them would discard meaningful signal.

In [ ]:
# check outlier with DBSCAN
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

X = df[['VehBCost', 'MMRCurrentRetailCleanPrice']].dropna()
X_scaled = StandardScaler().fit_transform(X)

db = DBSCAN(eps=0.5, min_samples=10).fit(X_scaled)
labels = db.labels_

# check target among DBSCAN outliers
df_temp = df.loc[X.index].assign(dbscan_label=labels)
outliers = df_temp[df_temp['dbscan_label'] == -1]['IsBadBuy']
display(pd.crosstab(outliers, columns='count'))

# visualization
plt.figure(figsize=(6,4))
sns.scatterplot(x=X['VehBCost'], y=X['MMRCurrentRetailCleanPrice'], hue=(labels==-1))
plt.title('DBSCAN Outliers (label = -1)')

In [ ]:
# Notes:
# DBSCAN outliers show a similar or higher BadBuy proportion, so removing them would also discard meaningful signal.

In [ ]:
# remove only economically absurd values, since statistical outliers still contain meaningful signal for BadBuy prediction


def remove_ilogical_rows(df):
    df = df.copy()

    # Warranty cost cannot exceed purchase price
    df = df[df['WarrantyCost'] <= df['VehBCost']]

    return df

In [ ]:
# remove_ilogical_rows TRAIN
print ('Shape for removing ilogical value:', '\nfeatures_train:', features_train.shape, 
       '\ntarget_train:', target_train.shape)
print(pd.crosstab(target_train, columns='count'))
      
features_train = remove_ilogical_rows(features_train)
target_train = target_train.loc[features_train.index]
      
print ('\nShape after removing ilogical value:', '\nfeatures_train:', features_train.shape, 
       '\ntarget_train:', target_train.shape)
print(pd.crosstab(target_train, columns='count'))

In [ ]:
# Notes:
# The removal of illogical rows affected both classes proportionally.  
# Before cleaning, the target distribution was 88% vs. 12%; after cleaning, it remained virtually unchanged (88% vs. 12%).
# This indicates that the filtering rules did not introduce class bias.

## Recample

In [ ]:
# recample with DecisionTreeClassifier

from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

from imblearn.pipeline import Pipeline

from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score

oversampler = RandomOverSampler(random_state=42)
undersampler = RandomUnderSampler(random_state=42)
smotesampler = SMOTE(random_state=42)

tree_clf = DecisionTreeClassifier(random_state=42)

key_cols = ['VehOdo', 'VehBCost', 'WarrantyCost', 'MMRCurrentRetailCleanPrice', 'VehicleAge']
features_key_train = features_train[key_cols]
features_key_test = features_test[key_cols]

search_space = {'estimator__max_depth': range(2, 16, 2),
                'estimator__class_weight': [None, 'balanced']}

samplers = [('oversampling', oversampler),
            ('undersampling', undersampler),
            ('class_weights', 'passthrough'),
            ('smote', smotesampler)]

results = []

for name, sampler in samplers:

    pipe_imb = Pipeline([('sampler', sampler),
                        ('estimator', tree_clf)])

    grid = GridSearchCV(estimator=pipe_imb,
                        param_grid=search_space,
                        n_jobs=-1,
                        cv=5,
                        scoring='f1')

    grid.fit(features_key_train, target_train)

    #evaluation
    target_pred = grid.best_estimator_.predict(features_key_test)

    precision = precision_score(target_test, target_pred)
    recall = recall_score(target_test, target_pred)

    print(name.upper())
    print("Cross‑validation best F1:", grid.best_score_)
    print("precision:", precision)
    print("recall:", recall)
    print(grid.best_estimator_)
    print('\n\n')
    
    results.append({
        'name': name,
        'precision': precision,
        'recall': recall
    })

pd.DataFrame(results)


In [ ]:
# Notes:
# oversampling, undersampling, and class‑weight balancing all produce nearly identical results  
# the F1‑score, precision, and recall remain unchanged, indicating that the simple decision tree (max_depth=2) 
# does not meaningfully benefit from altering the class distribution
# SMOTE performs slightly worse, especially in terms of recall, and therefore offers no advantage in this baseline setup